In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [16]:
def compute_binary_metrics(
    df,
    y_true_col="y_true",
    y_pred_col="y_pred",
    prob_col="prob_class_1",
    positive_class=1
):
    """
    Compute binary classification metrics for one dataframe subset.

    Returns one row as a pandas Series.
    """

    if df.empty:
        return pd.Series({
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "n_samples": 0
        })

    y_true = df[y_true_col]
    y_pred = df[y_pred_col]

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=positive_class,
        zero_division=0
    )

    # AUC needs probability scores and both classes must exist in y_true
    if prob_col in df.columns and y_true.nunique() == 2:
        auc = roc_auc_score(y_true, df[prob_col])
    else:
        auc = np.nan

    return pd.Series({
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "n_samples": len(df)
    })
def compute_metrics_by_group(
    df,
    group_cols,
    y_true_col="y_true",
    y_pred_col="y_pred",
    prob_col="prob_class_1",
    positive_class=1
):
    """
    Compute binary classification metrics for each group.

    Example:
        compute_metrics_by_group(
            mono_eng,
            group_cols=["feature_set", "model"]
        )
    """

    rows = []

    for group_values, group_df in df.groupby(group_cols):
        metrics = compute_binary_metrics(
            group_df,
            y_true_col=y_true_col,
            y_pred_col=y_pred_col,
            prob_col=prob_col,
            positive_class=positive_class
        )

        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        group_info = dict(zip(group_cols, group_values))
        row = {**group_info, **metrics.to_dict()}
        rows.append(row)

    return pd.DataFrame(rows)

In [3]:
results_df = pd.read_csv(r"D:\masteruwefduyqeahfdqe\ASR-project\model_predictions_final.csv")
results_df.head()

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
0,014-2,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.364486,0.635514
1,024-1,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.370597,0.629403
2,024-2,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.340176,0.659824
3,043-0,English,unknown,SVM,all,mono_english,mono,English,English,1,0,0.583802,0.416198
4,046-0,English,unknown,SVM,all,mono_english,mono,English,English,1,0,0.508399,0.491601


In [6]:
results_df[['model', 'language', 'feature_set', 'experiment_name']].value_counts()

model         language  feature_set  experiment_name                  
SVM           English   subset_wav   train_mandarin_greek_test_english    551
MLP           English   wav_only     train_mandarin_greek_test_english    551
XGBoost       English   subset_cha   train_mandarin_greek_test_english    551
                        subset_wav   train_mandarin_greek_test_english    551
                        cha_only     train_mandarin_greek_test_english    551
                                                                         ... 
RandomForest  Greek     all          mono_greek                            17
                                     train_mandarin_english_greek          17
                        cha_only     mono_greek                            17
KNN           Greek     wav_only     train_mandarin_english_greek          17
MLP           Greek     subset_cha   train_mandarin_english_greek          17
Name: count, Length: 225, dtype: int64

In [7]:
mono_eng_all = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "English") & (results_df["feature_set"] == "all")
]
mono_eng_all

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
0,014-2,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.364486,0.635514
1,024-1,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.370597,0.629403
2,024-2,English,unknown,SVM,all,mono_english,mono,English,English,1,1,0.340176,0.659824
3,043-0,English,unknown,SVM,all,mono_english,mono,English,English,1,0,0.583802,0.416198
4,046-0,English,unknown,SVM,all,mono_english,mono,English,English,1,0,0.508399,0.491601
...,...,...,...,...,...,...,...,...,...,...,...,...,...
790,323-0,English,unknown,MLP,all,mono_english,mono,English,English,0,1,0.401313,0.598687
791,323-1,English,unknown,MLP,all,mono_english,mono,English,English,0,1,0.447158,0.552842
792,612-0,English,unknown,MLP,all,mono_english,mono,English,English,0,1,0.451495,0.548505
793,686-0,English,unknown,MLP,all,mono_english,mono,English,English,0,0,0.683349,0.316651


In [ ]:
mono_eng_all.to_csv("mono_english_all_features_prediction.csv", index=False)

In [17]:
mono_eng_all_metrics = compute_metrics_by_group(mono_eng_all, group_cols=["model"])
mono_eng_all_metrics

,model,accuracy,balanced_accuracy,precision,recall,f1,auc,n_samples
0,KNN,0.603774,0.603180,0.634146,0.611765,0.622754,0.615819,159.0
1,MLP,0.547170,0.529253,0.553719,0.788235,0.650485,0.628458,159.0
2,RandomForest,0.641509,0.631479,0.634615,0.776471,0.698413,0.701749,159.0
3,SVM,0.610063,0.600318,0.611650,0.741176,0.670213,0.674086,159.0
4,XGBoost,0.584906,0.580286,0.604396,0.647059,0.625000,0.667250,159.0


In [9]:
mono_mandarin_all = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "Mandarin") & (results_df["feature_set"] == "all")
]
mono_mandarin_all

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
795,016_Daddy,Mandarin,unknown,SVM,all,mono_mandarin,mono,Mandarin,Mandarin,1,0,0.535302,0.464698
796,016_market,Mandarin,unknown,SVM,all,mono_mandarin,mono,Mandarin,Mandarin,1,0,0.693610,0.306390
797,016_park,Mandarin,unknown,SVM,all,mono_mandarin,mono,Mandarin,Mandarin,1,1,0.298505,0.701495
798,019_Daddy,Mandarin,unknown,SVM,all,mono_mandarin,mono,Mandarin,Mandarin,1,1,0.289782,0.710218
799,019_market,Mandarin,unknown,SVM,all,mono_mandarin,mono,Mandarin,Mandarin,1,1,0.482806,0.517194
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,032_market,Mandarin,unknown,MLP,all,mono_mandarin,mono,Mandarin,Mandarin,0,0,0.609316,0.390684
1196,032_park,Mandarin,unknown,MLP,all,mono_mandarin,mono,Mandarin,Mandarin,0,0,0.525107,0.474893
1197,062_Daddy,Mandarin,unknown,MLP,all,mono_mandarin,mono,Mandarin,Mandarin,0,0,0.530118,0.469882
1198,062_market,Mandarin,unknown,MLP,all,mono_mandarin,mono,Mandarin,Mandarin,0,1,0.422019,0.577981


In [ ]:
mono_mandarin_all.to_csv("mono_mandarin_all_features_prediction.csv", index=False)

In [18]:
mono_mandarin_metrics = compute_metrics_by_group(mono_mandarin_all, group_cols=["model"])
mono_mandarin_metrics

,model,accuracy,balanced_accuracy,precision,recall,f1,auc,n_samples
0,KNN,0.518519,0.514652,0.530612,0.619048,0.571429,0.583333,81.0
1,MLP,0.506173,0.502747,0.520833,0.595238,0.555556,0.613553,81.0
2,RandomForest,0.518519,0.514652,0.530612,0.619048,0.571429,0.559829,81.0
3,SVM,0.469136,0.464286,0.490196,0.595238,0.537634,0.473138,81.0
4,XGBoost,0.592593,0.587912,0.588235,0.714286,0.645161,0.529304,81.0


In [11]:
mono_greek_all = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "Greek") & (results_df["feature_set"] == "all")
]
mono_greek_all

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
1200,13_p,Greek,unknown,SVM,all,mono_greek,mono,Greek,Greek,1,1,0.190080,0.809920
1201,16_p,Greek,unknown,SVM,all,mono_greek,mono,Greek,Greek,1,1,0.173218,0.826782
1202,19_p,Greek,unknown,SVM,all,mono_greek,mono,Greek,Greek,1,1,0.194539,0.805461
1203,21_p,Greek,unknown,SVM,all,mono_greek,mono,Greek,Greek,1,1,0.226480,0.773520
1204,25_p,Greek,unknown,SVM,all,mono_greek,mono,Greek,Greek,1,1,0.198265,0.801735
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1280,52_p,Greek,unknown,MLP,all,mono_greek,mono,Greek,Greek,1,1,0.414315,0.585685
1281,5_p,Greek,unknown,MLP,all,mono_greek,mono,Greek,Greek,1,1,0.446465,0.553535
1282,10,Greek,unknown,MLP,all,mono_greek,mono,Greek,Greek,0,1,0.498184,0.501816
1283,5,Greek,unknown,MLP,all,mono_greek,mono,Greek,Greek,0,1,0.387275,0.612725


In [ ]:
mono_greek_all.to_csv("mono_greek_all_features_predictins.csv", index=False)

In [19]:
mono_greek_metrics = compute_metrics_by_group(mono_greek_all, group_cols=["model"])
mono_greek_metrics

,model,accuracy,balanced_accuracy,precision,recall,f1,auc,n_samples
0,KNN,0.823529,0.500000,0.823529,1.000000,0.903226,0.333333,17.0
1,MLP,0.705882,0.428571,0.800000,0.857143,0.827586,0.571429,17.0
2,RandomForest,0.823529,0.500000,0.823529,1.000000,0.903226,0.523810,17.0
3,SVM,0.823529,0.500000,0.823529,1.000000,0.903226,0.166667,17.0
4,XGBoost,0.764706,0.464286,0.812500,0.928571,0.866667,0.761905,17.0


In [20]:
mono_mandarin_metrics.to_csv("mono_mandarin_all_features_metrics.csv", index=False)
mono_greek_metrics.to_csv("mono_greek_all_features_metrics.csv", index=False)
mono_eng_all_metrics.to_csv("mono_english_all_features_metrics.csv", index=False)

In [21]:
mono_eng_cha = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "English") & (results_df["feature_set"] == "cha_only")
]
mono_eng_cha

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
6890,014-2,English,unknown,SVM,cha_only,mono_english,mono,English,English,1,1,0.218732,0.781268
6891,024-1,English,unknown,SVM,cha_only,mono_english,mono,English,English,1,1,0.454684,0.545316
6892,024-2,English,unknown,SVM,cha_only,mono_english,mono,English,English,1,1,0.198330,0.801670
6893,043-0,English,unknown,SVM,cha_only,mono_english,mono,English,English,1,1,0.368840,0.631160
6894,046-0,English,unknown,SVM,cha_only,mono_english,mono,English,English,1,1,0.439617,0.560383
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7680,323-0,English,unknown,MLP,cha_only,mono_english,mono,English,English,0,1,0.377481,0.622519
7681,323-1,English,unknown,MLP,cha_only,mono_english,mono,English,English,0,1,0.404353,0.595647
7682,612-0,English,unknown,MLP,cha_only,mono_english,mono,English,English,0,1,0.313917,0.686083
7683,686-0,English,unknown,MLP,cha_only,mono_english,mono,English,English,0,0,0.691166,0.308834


In [22]:
mono_eng_cha_metrics = compute_metrics_by_group(mono_eng_cha, group_cols=["model"])
mono_eng_cha_metrics

,model,accuracy,balanced_accuracy,precision,recall,f1,auc,n_samples
0,KNN,0.603774,0.598808,0.619565,0.670588,0.644068,0.624483,159.0
1,MLP,0.622642,0.600715,0.595420,0.917647,0.722222,0.669157,159.0
2,RandomForest,0.616352,0.607075,0.617647,0.741176,0.673797,0.693164,159.0
3,SVM,0.610063,0.600318,0.611650,0.741176,0.670213,0.653895,159.0
4,XGBoost,0.635220,0.629094,0.642105,0.717647,0.677778,0.684897,159.0


In [23]:
mono_eng_cha_metrics.to_csv("mono_english_cha_only_features_metrics.csv", index=False)

In [24]:
mono_greek_cha = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "Greek") & (results_df["feature_set"] == "cha_only")
]
mono_greek_cha

,sample_id,language,dataset,model,feature_set,experiment_name,strategy,train_languages,test_languages,y_true,y_pred,prob_class_0,prob_class_1
8090,13_p,Greek,unknown,SVM,cha_only,mono_greek,mono,Greek,Greek,1,1,0.196593,0.803407
8091,16_p,Greek,unknown,SVM,cha_only,mono_greek,mono,Greek,Greek,1,1,0.194092,0.805908
8092,19_p,Greek,unknown,SVM,cha_only,mono_greek,mono,Greek,Greek,1,1,0.197597,0.802403
8093,21_p,Greek,unknown,SVM,cha_only,mono_greek,mono,Greek,Greek,1,1,0.194399,0.805601
8094,25_p,Greek,unknown,SVM,cha_only,mono_greek,mono,Greek,Greek,1,1,0.199963,0.800037
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8170,52_p,Greek,unknown,MLP,cha_only,mono_greek,mono,Greek,Greek,1,1,0.259914,0.740086
8171,5_p,Greek,unknown,MLP,cha_only,mono_greek,mono,Greek,Greek,1,1,0.244166,0.755834
8172,10,Greek,unknown,MLP,cha_only,mono_greek,mono,Greek,Greek,0,1,0.314531,0.685469
8173,5,Greek,unknown,MLP,cha_only,mono_greek,mono,Greek,Greek,0,1,0.338486,0.661514


In [25]:
mono_greek_cha_metrics = compute_metrics_by_group(mono_greek_cha, group_cols=["model"])
mono_greek_cha_metrics

,model,accuracy,balanced_accuracy,precision,recall,f1,auc,n_samples
0,KNN,0.823529,0.5,0.823529,1.0,0.903226,0.500000,17.0
1,MLP,0.823529,0.5,0.823529,1.0,0.903226,0.833333,17.0
2,RandomForest,0.823529,0.5,0.823529,1.0,0.903226,0.761905,17.0
3,SVM,0.823529,0.5,0.823529,1.0,0.903226,0.619048,17.0
4,XGBoost,0.823529,0.5,0.823529,1.0,0.903226,0.833333,17.0


In [26]:
mono_greek_cha_metrics.to_csv("mono_greek_cha_only_features_metrics.csv", index=False)

In [27]:
mono_mandarin_cha = results_df[
    (results_df["strategy"] == "mono") & (results_df["language"] == "Mandarin") & (results_df["feature_set"] == "cha_only")
]
mono_mandarin_cha_metrics = compute_metrics_by_group(mono_mandarin_cha, group_cols=["model"])
mono_mandarin_cha_metrics.to_csv("mono_mandarin_cha_only_features_metrics.csv", index=False)

In [30]:
for language in ["English", "Greek", "Mandarin"]:
    for feature_set in ["all", "cha_only", "wav_only", "subset_wav","subset_cha"]:
        for strategy in ["mono"]:
            # subset = results_df[
            #     (results_df["strategy"] == strategy) &
            #     (results_df["language"] == language) &
            #     (results_df["feature_set"] == feature_set)
            # ]
            print(f"{strategy}_{language.lower()}_{feature_set}_features_metrics.csv")
            # metrics = compute_metrics_by_group(subset, group_cols=["model"])
            # metrics.to_csv(f"{strategy}_{language.lower()}_{feature_set}_features_metrics.csv", index=False)

mono_english_all_features_metrics.csv
mono_english_cha_only_features_metrics.csv
mono_english_wav_only_features_metrics.csv
mono_english_subset_wav_features_metrics.csv
mono_english_subset_cha_features_metrics.csv
mono_greek_all_features_metrics.csv
mono_greek_cha_only_features_metrics.csv
mono_greek_wav_only_features_metrics.csv
mono_greek_subset_wav_features_metrics.csv
mono_greek_subset_cha_features_metrics.csv
mono_mandarin_all_features_metrics.csv
mono_mandarin_cha_only_features_metrics.csv
mono_mandarin_wav_only_features_metrics.csv
mono_mandarin_subset_wav_features_metrics.csv
mono_mandarin_subset_cha_features_metrics.csv
